In [1]:
!nvidia-smi

Tue Oct 20 19:49:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.86                 Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   42C    P8              1W /  105W |      32MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 networkx openai pydantic tiktoken


BƯỚC 3: Embedding biến câu thành vector

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

docs = [
    "Retrieval-augmented generation combines a retriever with a generator to answer questions using external documents.",
    "We propose LoRA, a low-rank adaptation method that fine-tunes large language models with few trainable parameters.",
    "Dense passage retrieval encodes questions and passages into vectors and finds nearest neighbours.",
    "We study convolutional networks for image classification on ImageNet.",
    "BM25 is a lexical ranking function based on term frequency and document length.",
    "Reinforcement learning from human feedback aligns language models with human preferences.",
]
D = model.encode(docs, normalize_embeddings=True)          # shape (6, 768)
PREFIX = "Represent this sentence for searching relevant passages: "

def dense_rank(query):
    q = model.encode([PREFIX + query], normalize_embeddings=True)
    scores = (D @ q.T).ravel()
    for i in np.argsort(-scores):
        print(f"{scores[i]:.3f}  [{i}] {docs[i][:60]}")

dense_rank("how can a language model use external documents when answering?")
print()
dense_rank("cheap ways to adapt a huge model to a new task")
print()
dense_rank("LoRA")

c:\Users\ADMIN\anaconda3\envs\env_AI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 363.54it/s]


0.721  [0] Retrieval-augmented generation combines a retriever with a g
0.592  [5] Reinforcement learning from human feedback aligns language m
0.582  [2] Dense passage retrieval encodes questions and passages into 
0.576  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.560  [4] BM25 is a lexical ranking function based on term frequency a
0.462  [3] We study convolutional networks for image classification on 

0.647  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.544  [3] We study convolutional networks for image classification on 
0.499  [0] Retrieval-augmented generation combines a retriever with a g
0.493  [2] Dense passage retrieval encodes questions and passages into 
0.489  [5] Reinforcement learning from human feedback aligns language m
0.433  [4] BM25 is a lexical ranking function based on term frequency a

0.602  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.418  [2] Dense passage retrieval encodes questions and passa

BƯỚC 4: BM25 (tìm theo từ kkhóa)

In [4]:
from rank_bm25 import BM25Okapi
import re

def tok(s):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())

bm25 = BM25Okapi([tok(d) for d in docs])

def bm25_rank(query):
    scores = bm25.get_scores(tok(query))
    for i in np.argsort(-scores):
        print(f"{scores[i]:.3f}  [{i}] {docs[i][:60]}")

bm25_rank("how can a language model use external documents when answering?")
print()
bm25_rank("cheap ways to adapt a huge model to a new task")
print()
bm25_rank("LoRA")


2.481  [0] Retrieval-augmented generation combines a retriever with a g
0.625  [5] Reinforcement learning from human feedback aligns language m
0.526  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.000  [2] Dense passage retrieval encodes questions and passages into 
0.000  [3] We study convolutional networks for image classification on 
0.000  [4] BM25 is a lexical ranking function based on term frequency a

2.481  [0] Retrieval-augmented generation combines a retriever with a g
0.000  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.000  [2] Dense passage retrieval encodes questions and passages into 
0.000  [3] We study convolutional networks for image classification on 
0.000  [4] BM25 is a lexical ranking function based on term frequency a
0.000  [5] Reinforcement learning from human feedback aligns language m

1.162  [1] We propose LoRA, a low-rank adaptation method that fine-tune
0.000  [0] Retrieval-augmented generation combines a retriever

Buoc 5: chay RAG tren 500 paper 

In [5]:
import json, os, numpy as np, faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
# chạy local: không dùng Colab Secrets

os.environ["LLM_API_KEY"] = os.environ.get("LLM_API_KEY", "")  # đặt key bằng biến môi trường, KHÔNG dán vào notebook  
os.environ["LLM_MODEL"] = "gpt-6-astra"   
os.environ["LLM_BASE_URL"] = "https://api.miraiapi.com/v1" 

BASE = r"D:\NLP\BTL\btl-nlp-hydrid-graph-rag-ArXiv"   # cùng giá trị với ô 3 của bước 1
papers = [json.loads(l) for l in open(f"{BASE}/data/processed/pilot_papers.jsonl")]
texts = [f"{p['title']}. {p['abstract']}" for p in papers]

enc = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cpu")   # 500 abstract: CPU đủ nhanh, nhường VRAM cho Ollama
emb = enc.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
PREFIX = "Represent this sentence for searching relevant passages: "

def search(q, k=5):
    qv = enc.encode([PREFIX + q], normalize_embeddings=True).astype("float32")
    D, I = index.search(qv, k)
    return [(int(i), float(d)) for i, d in zip(I[0], D[0])]

client = OpenAI(base_url=os.environ.get("LLM_BASE_URL"), api_key=os.environ["LLM_API_KEY"])
PROMPT = """Answer the question using ONLY the numbered sources.
Cite every claim with [n]. If the sources are insufficient, reply exactly: Insufficient information.

Sources:
{sources}

Question: {question}
Answer:"""

def rag(question, k=5):
    hits = search(question, k)
    sources = "\n\n".join(
        f"[{n}] {papers[i]['title']} (arXiv:{papers[i]['paper_id']})\n{texts[i]}"
        for n, (i, _) in enumerate(hits, 1))
    r = client.chat.completions.create(
        model=os.environ["LLM_MODEL"], temperature=0,
        messages=[{"role": "user", "content": PROMPT.format(sources=sources, question=question)}])
    return r.choices[0].message.content, hits, r.usage

Batches: 100%|██████████| 8/8 [01:21<00:00, 10.19s/it]


In [6]:
# Danh sách câu hỏi test tay
questions = [
    "How can iterative evaluation improve the reliability of autonomous LLM agents?",
    "How does metal doping influence the stability of silicon-based nano-structures?",
    "What are the macro-scale impacts of thermal energy transitions in gases?",
    "How can quantum computing redefine the underlying molecular simulation algorithms for inorganic chemical synthesis?"
]
# Chạy thử nghiệm từng câu hỏi qua hệ thống RAG của bạn
for i, q in enumerate(questions, 1):
    print(f"\n================ QUERY {i} ================")
    print(f"Câu hỏi: {q}\n")
    
    # Gọi hàm RAG đã định nghĩa của bạn
    answer, hits, usage = rag(q, k=5)
    
    print("----- KẾT QUẢ TỪ LLM -----")
    print(answer)
    print("\n--------------------------")
    print(f"Top 5 IDs bài báo tìm được: {[int(hit[0]) for hit in hits]}")
    print(f"Token usage: {usage}")



================ QUERY 1 ================
Câu hỏi: How can iterative evaluation improve the reliability of autonomous LLM agents?

----- KẾT QUẢ TỪ LLM -----
## Iterative evaluation as a reliability mechanism

Iterative evaluation helps an autonomous LLM agent by checking work *during* the rollout — at the level of individual reasoning steps, candidate solutions, and guarded actions — and feeding those verdicts back to steer the next step.

**1. Step-level verification of multi-step reasoning.** Test-time scaling works by sampling multiple variants of intermediate reasoning steps, verifying their correctness, and selecting the best steps for continuation [2]. Verification need not be expensive: a lightweight transformer probe (fewer than 10M parameters) reads the *internal states of a frozen LLM* during generation to estimate each step's credibility, with annotation supplied either by a larger LLM (e.g., DeepSeek-R1) or self-supervised by the original model itself. Such probes match o

In [7]:
import random
random.seed(0)

def title_recall_at_1(n=100):
    ok = 0
    for i in random.sample(range(len(papers)), n):
        top = [j for j, _ in search(papers[i]["title"], 1)]
        ok += (top[0] == i)
    return ok / n

print(title_recall_at_1()) 

1.0


In [8]:
def evaluate(qa, k=10):
    """qa: danh sách (câu hỏi, chỉ số paper gold)."""
    ranks = []
    for q, gold in qa:
        ids = [i for i, _ in search(q, k)]
        ranks.append(ids.index(gold) + 1 if gold in ids else None)
    recall = sum(r is not None for r in ranks) / len(ranks)
    mrr = sum(1 / r for r in ranks if r) / len(ranks)
    return recall, mrr, ranks

Bước 7: Knowledge Graph bằng tay

In [9]:
import networkx as nx

G = nx.DiGraph()
triples = [
    ("P1", "USES", "DPR"), ("P1", "USES", "BM25"),
    ("P2", "USES", "DPR"), ("P2", "USES", "ColBERT"),
    ("P3", "PROPOSES", "ColBERT"),
    ("P4", "USES", "BM25"),
]
for s, rel, o in triples:
    G.add_edge(s, o, rel=rel)

# Hop 1: paper nào DÙNG DPR?
using_dpr = [p for p in G.predecessors("DPR") if G.edges[p, "DPR"]["rel"] == "USES"]
# Hop 2: những paper đó còn DÙNG phương pháp nào khác?
co_methods = {m for p in using_dpr for m in G.successors(p)
              if m != "DPR" and G.edges[p, m]["rel"] == "USES"}
print(using_dpr, co_methods)     # ['P1', 'P2'] và {'BM25', 'ColBERT'}

['P1', 'P2'] {'ColBERT', 'BM25'}


Bước 8: Trích xuất quan hệ bằng LLM và đo chất lượng

In [ ]:
import os, json, random, csv
BASE = r'D:\NLP\BTL\btl-nlp-hydrid-graph-rag-ArXiv'   # cùng giá trị với bước 1
papers = [json.loads(l) for l in open(f'{BASE}/data/processed/pilot_papers.jsonl')]
print(len(papers), 'paper | trường:', sorted(papers[0].keys()))   # cần có paper_id, title, abstract

random.seed(7)                      # seed cố định: chạy lại vẫn ra đúng 20 paper này
dev = random.sample(papers, 20)
os.makedirs(f'{BASE}/data/eval', exist_ok=True)
with open(f'{BASE}/data/eval/dev_extract.jsonl', 'w') as f:
    for p in dev:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

# File để đọc và gán nhãn. Dùng cột idx (1-20) thay vì paper_id: Google Sheets hay tự biến
# '2401.10100' thành số 2401.101 và làm hỏng id.
with open(f'{BASE}/data/eval/dev_extract_to_label.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['idx', 'title', 'abstract'])
    for i, p in enumerate(dev, 1):
        w.writerow([i, p['title'], p['abstract']])
print('Đã ghi 20 paper. Mở data/eval/dev_extract_to_label.csv để gán nhãn.')

500 paper | trường: ['abstract', 'authors', 'categories', 'date', 'paper_id', 'title']
Đã ghi 20 paper. Mở data/eval/dev_extract_to_label.csv để gán nhãn.


In [3]:
import os, json, random, csv
BASE = r'D:\NLP\BTL\btl-nlp-hydrid-graph-rag-ArXiv'   # cùng giá trị với bước 1
papers = [json.loads(l) for l in open(f'{BASE}/data/processed/pilot_papers.jsonl')]
print(len(papers), 'paper | trường:', sorted(papers[0].keys()))   # cần có paper_id, title, abstract

random.seed(42)                      # seed cố định: chạy lại vẫn ra đúng 20 paper này
dev = random.sample(papers, 20)
os.makedirs(f'{BASE}/data/eval', exist_ok=True)
with open(f'{BASE}/data/eval/dev_extract(1).jsonl', 'w') as f:
    for p in dev:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

# File để đọc và gán nhãn. Dùng cột idx (1-20) thay vì paper_id: Google Sheets hay tự biến
# '2401.10100' thành số 2401.101 và làm hỏng id.
with open(f'{BASE}/data/eval/dev_extract_to_label(1).csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.writer(f); w.writerow(['idx', 'title', 'abstract'])
    for i, p in enumerate(dev, 1):
        w.writerow([i, p['title'], p['abstract']])
print('Đã ghi 20 paper. Mở data/eval/dev_extract_to_label(1).csv để gán nhãn.')

500 paper | trường: ['abstract', 'authors', 'categories', 'date', 'paper_id', 'title']
Đã ghi 20 paper. Mở data/eval/dev_extract_to_label(1).csv để gán nhãn.


In [11]:
import os
import time
from openai import OpenAI
c = OpenAI(base_url=os.environ['LLM_BASE_URL'], api_key=os.environ['LLM_API_KEY'])
# bỏ qua liệt kê model vì không phải nhà cung cấp nào cũng hỗ trợ;          (với Ollama: dùng ollama list để xem tag)
t0 = time.time()
r = c.chat.completions.create(model=os.environ['LLM_MODEL'], temperature=0, max_tokens=64,
                              messages=[{'role': 'user', 'content': 'Say "ok" and nothing else.'}])
print(r.choices[0].message.content, '|', round(time.time() - t0, 1), 's |', r.usage.prompt_tokens, 'token vào,', r.usage.completion_tokens, 'token ra')

ok | 1.9 s | 3487 token vào, 3 token ra


In [15]:
import os, json, time, re
from typing import Literal
from pydantic import BaseModel, ValidationError
from openai import OpenAI
# chạy local: không dùng Colab Secrets

try:
    os.environ['LLM_API_KEY'] = os.environ['LLM_API_KEY']   # KeyError nếu chưa đặt biến môi trường
except Exception:
    raise SystemExit('Chưa đặt LLM_API_KEY trong biến môi trường (xem bước 1); nếu dùng Ollama dự phòng thì gán os.environ["LLM_API_KEY"] = "ollama"')
MODEL = os.environ.get('LLM_MODEL', 'gpt-6-astra')   # điền tên model của nhà cung cấp (hoặc đặt LLM_MODEL); ghi lại để tái lập
BASE_URL = os.environ.get('LLM_BASE_URL', 'https://api.miraiapi.com/v1')   # đổi theo nhà cung cấp bạn chọn (xem mục Chọn LLM bên dưới)
client = OpenAI(base_url=BASE_URL, api_key=os.environ['LLM_API_KEY'],
                max_retries=2, timeout=120)   # tự thử lại khi bị 429/503 (nghẽn)
JSON_MODE = False                     # để False: prompt + Pydantic vẫn ép JSON; thử True nếu nhà cung cấp hỗ trợ json_object và bạn hay gặp lỗi parse
PAUSE = 0                             # API trả phí thường để 0 (đã có retry khi gặp 429); tăng nếu hay bị 429; Gemini free cần khoảng 6 (RPM xem ở aistudio.google.com/rate-limit)
PROMPT_VERSION = 'v2'                 # đổi v2, v3 mỗi lần sửa prompt để so F1 giữa các vòng

import random
from openai import RateLimitError, APIStatusError, APIConnectionError

def create_with_retry(**kw):
    for attempt in range(8):
        try:
            r = client.chat.completions.create(**kw)
            time.sleep(PAUSE)
            return r
        except (RateLimitError, APIConnectionError):
            pass
        except APIStatusError as e:
            if e.status_code not in (429, 500, 502, 503, 504):
                raise                      # 400/401/404...: lỗi thật (sai key, sai tên model), không thử lại
        wait = min(60, 3 * 2 ** attempt) + random.random()
        print(f'  lỗi tạm thời (quá tải hoặc hết hạn mức phút), chờ {wait:.0f}s rồi thử lại ({attempt + 1}/8)')
        time.sleep(wait)
    raise RuntimeError('Hết lượt thử lại. Lỗi 429 kéo dài thường là hết hạn mức ngày (reset lúc nửa đêm giờ Thái Bình Dương). Chạy lại ô này sau, phần đã xong được giữ.')

class Relation(BaseModel):
    relation: Literal['PROPOSES', 'USES', 'ADDRESSES', 'EVALUATES_ON']
    target: str
    target_type: str = ''          # đừng ép Literal: model hay trả chữ thường hoặc nhãn khác
    evidence: str

TYPE_MAP = {'method': 'Method', 'model': 'Method', 'technique': 'Method', 'algorithm': 'Method',
            'task': 'Task', 'problem': 'Task',
            'dataset': 'Dataset', 'benchmark': 'Dataset', 'resource': 'Dataset'}

class Extraction(BaseModel):
    relations: list[Relation]

SYSTEM = '''You extract a knowledge graph from a scientific abstract.
Return JSON: {"relations": [{"relation", "target", "target_type", "evidence"}]}.
Relations (subject is always the paper):
- PROPOSES: the paper introduces a NEW method/model/technique (usually named by the authors).
- USES: the paper uses an EXISTING method as a component or backbone. Do NOT use it for methods only compared against.
- ADDRESSES: the task/problem the paper tackles.
- EVALUATES_ON: a named dataset or benchmark used for evaluation.
Rules: use the shortest standard name for target (e.g. "BERT", not "the BERT model").
"evidence" must be copied verbatim from the text. Do not infer anything not stated.
If nothing applies, return {"relations": []}.'''

def extract(paper):
    text = f"Title: {paper['title']}\nAbstract: {paper['abstract']}"
    resp = create_with_retry(
        model=MODEL, temperature=0,
        messages=[{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': text}],
        **({'response_format': {'type': 'json_object'}} if JSON_MODE else {}),
    )
    raw = (resp.choices[0].message.content or '').strip()   # content có thể rỗng nếu model từ chối
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)   # model có chế độ suy nghĩ hay in kèm khối <think>
    i, j = raw.find('{'), raw.rfind('}')
    if i != -1 and j > i:
        raw = raw[i:j + 1]                          # cắt đúng phần JSON, bỏ ``` và lời dẫn
    out = {'paper_id': paper['paper_id'], 'extractor': MODEL, 'prompt': PROMPT_VERSION,
           'usage': {'in': getattr(resp.usage, 'prompt_tokens', 0), 'out': getattr(resp.usage, 'completion_tokens', 0)}}
    try:
        ex = Extraction.model_validate_json(raw)
    except ValidationError:
        return {**out, 'relations': [], 'error': 'invalid_json', 'raw': raw}
    src = text.lower()
    kept = [r.model_dump() for r in ex.relations if r.evidence.lower() in src]   # bỏ quan hệ bịa evidence
    for r in kept:
        r['target_type'] = TYPE_MAP.get(r['target_type'].lower(), 'Other')
    return {**out, 'relations': kept, 'dropped_no_evidence': len(ex.relations) - len(kept)}

dev = [json.loads(l) for l in open(f'{BASE}/data/eval/dev_extract.jsonl')]
OUT2 = f'{BASE}/data/kg/dev_extract_out_{PROMPT_VERSION}.jsonl'
cache = {}
if os.path.exists(OUT2):                # đã có kết quả của vòng này thì không gọi lại, bị ngắt giữa chừng vẫn làm tiếp được
    cache = {json.loads(l)['paper_id']: json.loads(l) for l in open(OUT2) if l.strip()}
t0 = time.time(); n_new = 0
outs = []
for p in dev:
    if p['paper_id'] not in cache:
        cache[p['paper_id']] = extract(p); n_new += 1
        with open(OUT2, 'a') as f:
            f.write(json.dumps(cache[p['paper_id']], ensure_ascii=False) + '\n')
    outs.append(cache[p['paper_id']])
sec = (time.time() - t0) / max(n_new, 1)      # đã gồm thời gian nghỉ PAUSE

tin = sum(o['usage']['in'] for o in outs); tout = sum(o['usage']['out'] for o in outs)
print(f"{sec:.1f}s/paper | token vào {tin}, ra {tout} | JSON lỗi: {sum('error' in o for o in outs)}"
      f" | bị lọc vì thiếu evidence: {sum(o.get('dropped_no_evidence', 0) for o in outs)}")
print(f'Ước lượng 15.000 paper: {tin*750:,} token vào, {tout*750:,} token ra (nhân đơn giá hiện hành)')
for o in outs[:3]:
    print(o['paper_id'], [(r['relation'], r['target']) for r in o['relations']])

17.4s/paper | token vào 80181, ra 11463 | JSON lỗi: 0 | bị lọc vì thiếu evidence: 1
Ước lượng 15.000 paper: 60,135,750 token vào, 8,597,250 token ra (nhân đơn giá hiện hành)
2402.19294 [('ADDRESSES', 'failure mode diagnosis and prognostic analysis under unknown failure modes'), ('PROPOSES', 'failure mode diagnosis method using UMAP and time series-based clustering'), ('PROPOSES', 'monotonically constrained prognostic model'), ('USES', 'UMAP'), ('EVALUATES_ON', 'aircraft gas turbine engine dataset')]
2403.13128 [('PROPOSES', 'AdaFish'), ('ADDRESSES', 'parameter-efficient fine-tuning'), ('USES', 'generalized Fisher information matrix'), ('USES', 'Hessian matrix'), ('EVALUATES_ON', 'AdamW')]
2505.00026 [('ADDRESSES', 'Theory of Mind capabilities of Large Language Models'), ('EVALUATES_ON', 'story-based benchmarks')]


In [18]:
import csv, re
from collections import defaultdict
RELS = {'PROPOSES', 'USES'}            # vòng đầu chỉ chấm hai loại này

def norm(s): return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()
def triples(rels): return {(r['relation'], norm(r['target'])) for r in rels if r['relation'] in RELS}

def micro_prf(pred, gold, loose=False):
    """Micro P/R/F1 trên (relation, target) của mọi paper. loose=True: target chứa hoặc nằm trong target kia."""
    same = (lambda a, b: a[0] == b[0] and (a[1] in b[1] or b[1] in a[1])) if loose else (lambda a, b: a == b)
    tp_p = tp_g = n_p = n_g = 0
    for pid, g in gold.items():
        g, p = triples(g), triples(pred.get(pid, []))
        n_g += len(g); n_p += len(p)
        tp_p += sum(any(same(x, y) for y in g) for x in p)   # dự đoán đúng
        tp_g += sum(any(same(x, y) for y in p) for x in g)   # gold được tìm thấy
    P = tp_p / n_p if n_p else 0.0
    R = tp_g / n_g if n_g else 0.0
    return {'P': round(P, 3), 'R': round(R, 3), 'F1': round(2*P*R/(P+R), 3) if P + R else 0.0, 'n_pred': n_p, 'n_gold': n_g}

rows = defaultdict(list)
for r in csv.DictReader(open(f'{BASE}/data/eval/gold_dev20.csv', encoding='utf-8-sig')):
    rows[int(r['idx'])].append(r)
gold = {p['paper_id']: rows.get(i, []) for i, p in enumerate(dev, 1)}   # paper không có quan hệ vẫn được tính
pred = {o['paper_id']: o['relations'] for o in outs}

print('khớp chặt:', micro_prf(pred, gold))
print('khớp lỏng:', micro_prf(pred, gold, loose=True))
for i, p in enumerate(dev, 1):          # đọc từng lỗi, gán loại: tên viết khác / sai loại / bịa / bỏ sót
    g, q = triples(gold[p['paper_id']]), triples(pred[p['paper_id']])
    if g != q:
        print(f"[{i}] {p['paper_id']} | thiếu: {sorted(g - q)} | thừa: {sorted(q - g)}")

khớp chặt: {'P': 0.347, 'R': 0.548, 'F1': 0.425, 'n_pred': 49, 'n_gold': 31}
khớp lỏng: {'P': 0.469, 'R': 0.677, 'F1': 0.555, 'n_pred': 49, 'n_gold': 31}
[1] 2402.19294 | thiếu: [('PROPOSES', 'failure mode diagnosis method')] | thừa: [('PROPOSES', 'failure mode diagnosis method using umap and time series based clustering')]
[2] 2403.13128 | thiếu: [] | thừa: [('USES', 'generalized fisher information matrix'), ('USES', 'hessian matrix')]
[4] 2305.05813 | thiếu: [] | thừa: [('PROPOSES', 'taxonomy of change detection algorithms'), ('USES', 'deep learning'), ('USES', 'transformer')]
[5] 2502.21228 | thiếu: [('USES', 'wikipedia')] | thừa: []
[6] 2511.07879 | thiếu: [('USES', 'ner')] | thừa: [('USES', 'named entity recognition')]
[7] 2406.08414 | thiếu: [] | thừa: [('PROPOSES', 'llm driven objective discovery'), ('USES', 'exponential loss'), ('USES', 'llm'), ('USES', 'logistic loss')]
[8] 2408.00800 | thiếu: [('USES', 'sparql')] | thừa: [('USES', 'domain specific standards'), ('USES', 'large

In [17]:
# pip install "openai>=1.40"
import os
from openai import OpenAI

c = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
for m in sorted(x.id for x in c.models.list()):
    print(m)

ccmax-claude-opus-5.5
claude-fable-5
claude-fable-5.1
claude-opus-4.6
claude-opus-4.7
claude-opus-4.8
claude-opus-5
claude-opus-5.5
claude-sonnet-5
claude-sonnet-5.5
gemini-4-argon
gpt-5.6-luna
gpt-5.6-sol
gpt-5.6-terra
gpt-6-astra
gpt-6-sol
gpt-6.1-sol
swe-2
